<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Programu Operacyjnego Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

# Logistic regression

In this exercise you will train a logistic regression model via gradient descent in two simple scenarios.

The general setup is as follows:
* we are given a set of pairs $(x, y)$, where $x \in R^D$ is a vector of real numbers representing the features, and $y \in \{0,1\}$ is the target,
* for a given $x$ we model the probability of $y=1$ by $h(x):=g(w^Tx)$, where $g$ is the sigmoid function: $g(z) = \frac{1}{1+e^{-z}}$,
* to find the right $w$ we will optimize the so called logarithmic loss: $J(w) = -\frac{1}{n}\sum_{i=1}^n y_i \log{h(x_i)} + (1-y_i) \log{(1-h(x_i))}$,
* with the loss function in hand we can improve our guesses iteratively:
    * $w_j^{t+1} = w_j^{t} - \eta \cdot \frac{\partial J(w)}{\partial w_j}$

* we can end the process after some predefined number of epochs (or when the changes are no longer meaningful).

Let's start with the simplest example - linear separated points on a plane.

In [39]:
import numpy as np

np.random.seed(123)

# these parametrize the line
a = 0.3
b = -0.2
c = 0.001

# True/False mapping
def lin_rule(x, noise=0.):
    return a * x[0] + b * x[1] + c + noise < 0.

# Just for plotting
def get_y_fun(a, b, c):
    def y(x):
        return - x * a / b - c / b
    return y

lin_fun = get_y_fun(a, b, c)

In [40]:
# Training data

n = 500
range_points = 1
sigma = 0.05

X = range_points * 2 * (np.random.rand(n, 2) - 0.5)
y = [lin_rule(x, sigma * np.random.normal()) for x in X]
print(X[:10])
print(y[:10])

[[ 0.39293837 -0.42772133]
 [-0.54629709  0.10262954]
 [ 0.43893794 -0.15378708]
 [ 0.9615284   0.36965948]
 [-0.0381362  -0.21576496]
 [-0.31364397  0.45809941]
 [-0.12285551 -0.88064421]
 [-0.20391149  0.47599081]
 [-0.63501654 -0.64909649]
 [ 0.06310275  0.06365517]]
[np.False_, np.True_, np.False_, np.False_, np.False_, np.True_, np.False_, np.True_, np.True_, np.False_]


Let's plot the data.

In [41]:
import plotly.express as px

# plotly has a problem with coloring boolean values, hence stringify
# see https://community.plotly.com/t/plotly-express-scatter-color-not-showing/25962
fig = px.scatter(x=X[:, 0], y=X[:, 1], color=list(map(str, y)))
x_range = [np.min(X[:, 0]), np.max(X[:, 1])]
fig.add_scatter(x=x_range, y=list(map(lin_fun, x_range)), name='ground truth border')
fig.update_layout(xaxis_title="Feature x_0", yaxis_title="Feature x_1")
fig.show()

Now, let's implement and train a logistic regression model. You should obtain accuracy of at least 96%.

In [42]:
################################################################
# TODO: Implement logistic regression and compute its accuracy #

### IMPORTANT CHANGE TO X -- WE ARTIFICIALLY ADD 3RD DIMENSION TO EACH xi AND PUT IT TO 1
### TO MAKE BIAS CALCULATIONS INTO MATRIX SHORTHAND FORM
X = np.stack([X[:,0], X[:,1], np.ones(n)], axis=1)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Logistic regression training with gradient descent
def logistic_regression(X, y, lr=0.1, epochs=1000):
    n_samples, n_features = X.shape
    y = np.array(y, dtype=np.float64)
    w = np.zeros([n_features])
    print(w)
    losses = []

    for epoch in range(epochs):
        # Predictions
        z = np.dot(X, w)
        h = sigmoid(z)

        # Gradient
        grad = (1/n_samples) * np.matmul(X.T, (h - y))

        # Weight update
        w -= lr * grad

        loss = -np.mean(y*np.log(h) + (1 - y)*np.log(1 - h))
        losses.append(loss)
        # Optional: monitor loss every 100 epochs
        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d} | Loss: {loss:.4f}")

    return w, losses

# Train the model
w, losses = logistic_regression(X, y, lr=0.8, epochs=1000)
print("Learned weights:", w)

# Compute accuracy
pred_probs = sigmoid(np.dot(X, w))
preds = pred_probs >= 0.5
accuracy = np.mean(preds == y)

print(f"Training accuracy: {accuracy * 100:.2f}%")
################################################################

[0. 0. 0.]
Epoch    0 | Loss: 0.6931
Epoch  100 | Loss: 0.1850
Epoch  200 | Loss: 0.1495
Epoch  300 | Loss: 0.1340
Epoch  400 | Loss: 0.1250
Epoch  500 | Loss: 0.1191
Epoch  600 | Loss: 0.1147
Epoch  700 | Loss: 0.1115
Epoch  800 | Loss: 0.1089
Epoch  900 | Loss: 0.1068
Learned weights: [-9.79147627  6.75273067  0.04530805]
Training accuracy: 96.60%


In [43]:
# Compute accuracy
scale = w[0]/a
pred_probs = sigmoid(np.dot(X, np.array([scale*a,scale*b,scale*c])))
preds = pred_probs >= 0.5
accuracy = np.mean(preds == y)

print(f"Training accuracy: {accuracy * 100:.2f}%")

Training accuracy: 96.40%


In [44]:
fig = px.line(y=losses, labels={'y':'loss'})
fig.show()

Let's visually asses our model. We can do this by using our estimates for $a,b,c$.

In [45]:
#################################################################
# TODO: Pass your estimates for a,b,c to the get_y_fun function #
#################################################################
lin_fun2 = get_y_fun(w[0], w[1], w[2])

fig = px.scatter(x=X[:, 0], y=X[:, 1], color=list(map(str, y)))
x_range = [np.min(X[:, 0]), np.max(X[:, 1])]
fig.add_scatter(x=x_range, y=list(map(lin_fun, x_range)), name='ground truth border')
fig.add_scatter(x=x_range, y=list(map(lin_fun2, x_range)), name='estimated border')
fig.show()

Let's now complicate the things a little bit and make our next problem nonlinear.

In [46]:
# Parameters of the ellipse
s1 = 1.
s2 = 2.
r = 0.75
m1 = 0.15
m2 = 0.125

# 0/1 mapping, checks whether we are inside the ellipse
def circle_rule(x, y, noise=0.):
    return 1 if s1 * (x - m1) ** 2 + s2 * (y - m2) ** 2 + noise < r ** 2 else 0

In [47]:
# Training data

n = 500
range_points = 1

sigma = 0.1

X = range_points * 2 * (np.random.rand(n, 2) - 0.5)

y = [circle_rule(x, y, sigma * np.random.normal()) for x, y in X]

print(X[:10])
print(y[:10])

[[ 0.18633789  0.87560968]
 [-0.81999293  0.61838609]
 [ 0.22604784  0.28001611]
 [ 0.9846182  -0.35783437]
 [-0.27962406  0.07170775]
 [ 0.2501677  -0.37650776]
 [ 0.41264707 -0.8357508 ]
 [-0.61039043 -0.97349628]
 [ 0.49924022  0.89579621]
 [ 0.537422   -0.65425777]]
[0, 0, 1, 0, 1, 1, 0, 0, 0, 0]


Let's plot the data.

In [48]:
import plotly.graph_objects as go

fig = px.scatter(x=X[:, 0], y=X[:, 1], color=list(map(str, y)))

xgrid = np.arange(np.min(X[:, 0]), np.max(X[:, 0]), 0.003)
ygrid = np.arange(np.min(X[:, 1]), np.max(X[:, 1]), 0.003)
fig.update_layout(xaxis_title="Feature x_0", yaxis_title="Feature x_1")
contour =  go.Contour(
        z=np.vectorize(circle_rule)(*np.meshgrid(xgrid, ygrid, indexing="xy")),
        x=xgrid,
        y=ygrid
    )
fig.add_trace(contour)
fig.show()

Now, let's train a logistic regression model to tackle this problem. Note that we now need a nonlinear decision boundary. You should obtain accuracy of at least 90%.

Hint:
<sub><sup><sub><sup><sub><sup>
Use feature engineering.
</sup></sub></sup></sub></sup></sub>

In [51]:
!pip install --upgrade numpy

In [50]:
################################################################
# TODO: Implement logistic regression and compute its accuracy #

# IMPORTANT: The decision looks like
# ax0^2 + bx1^2 + cx0 + dx1 + e < 0 for some constants
# in order to model it with logistic regression, we will have to introduce
# parameters x0, x1, x0^2, x1^2 so that the problem is linear in them
print(X.shape)
L = np.unstack(X, axis=1)
print(L)
#X = np.stack([X, X*X], axis=1)
print(X.shape)
X[:4]
################################################################

(500, 2)


AttributeError: module 'numpy' has no attribute 'unstack'

Let's visually asses our model.

Contrary to the previous scenario, converting our weights to parameters of the ground truth curve may not be straightforward. It's easier to just provide predictions for a set of points in $R^2$.

In [ ]:
h = .02

xgrid = np.arange(np.min(X[:, 0]), np.max(X[:, 0]), h)
ygrid = np.arange(np.min(X[:, 1]), np.max(X[:, 1]), h)

xx, yy = np.meshgrid(xgrid, ygrid, indexing="ij")
X_plot = np.c_[xx.ravel(), yy.ravel()]

print(X_plot.shape)

_X = np.concatenate([X_plot, X_plot**2], axis=1)

preds = logistic_regression(_w, _b, _X)
print(preds.shape)


In [ ]:
fig = px.scatter(x=X[:, 0], y=X[:, 1], color=list(map(str, y)))

xx, yy = np.meshgrid(xgrid, ygrid, indexing="ij")

contour = go.Contour(z=preds.reshape(len(xgrid), len(ygrid)), x=xgrid, y=ygrid)
fig.add_trace(contour)
fig.show()

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>